# Viewing q-matroid classification results

- View the canonical representative of each isomorphism class from the saved classification results.
- Select the data using `q` (field order), `n` (dimension of the ambient space), and `k` (rank of the q-matroid).
- The included results cover:
  - All ranks in dimensions 0 through 5 over $\mathrm{GF}(2)$.
  - All ranks in dimensions 0 through 4 over $\mathrm{GF}(3)$.

In [1]:
from collections import Counter
from pathlib import Path
from time import perf_counter

from sage.all import GF, VectorSpace, table

from qmatroid.finite_geometry import (
    ReversedLexicographicOrderOfSubspaces,
    get_subspace_tables,
)
from qmatroid.q_matroid_enumeration import load_q_matroids

q = 2
n = 5
k = 2

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent
path = project_root / "results" / f"q{q}_dimension{n}_rank{k}.txt"
tables = get_subspace_tables(q, n)
Ms = load_q_matroids(path, q=q, n=n)

display(table(
    [
        ("file", str(path.relative_to(project_root))),
        ("field order", q),
        ("ambient dimension", n),
        ("rank", k),
        ("stored representatives", len(Ms)),
    ],
    header_row=["item", "value"],
))

  item                     value
├────────────────────────┼─────────────────────────────────┤
  file                     results/q2_dimension5_rank2.txt
  field order              2
  ambient dimension        5
  rank                     2
  stored representatives   94

## List of all isomorphism classes

- The following values are displayed for each canonical representative:
  - Isomorphism class number.
  - Number of bases and number of hyperplanes.
  - Order of the projective automorphism group (acting on one-dimensional subspaces).

In [2]:
started = perf_counter()
overview = [
    (
        i,
        len(M.basis_ids(tables)),
        len(M.hyperplane_ids(tables)),
        M.projective_automorphism_group(tables).order(),
    )
    for i, M in enumerate(Ms, start=1)
]
display(table(
    overview,
    header_row=[
        "class",
        "number of bases",
        "number of hyperplanes",
        "projective automorphism order",
    ],
))
print(f"Computed the overview in {perf_counter() - started:.3f} seconds.")

  class   number of bases   number of hyperplanes   projective automorphism order
├───────┼─────────────────┼───────────────────────┼───────────────────────────────┤
  1       64                3                       64512
  2       96                5                       9216
  3       112               9                       21504
  4       120               17                      322560
  5       112               7                       64512
  6       120               5                       5760
  7       124               7                       1152
  8       128               9                       576
  9       132               11                      1152
  10      136               13                      9216
  11      144               17                      288
  12      142               13                      6
  13      142               13                      6
  14      143               15                      2
  15      144               17            

Computed the overview in 1.005 seconds.


## A single canonical representative

- Set `selected` to an isomorphism class number from the list (starting at 1).
- The selected canonical representative is stored in `M` and used in the subsequent automorphism group and basis encoding calculations.

In [3]:
selected = 1
M = Ms[selected - 1]
V = VectorSpace(GF(q), n)
order = ReversedLexicographicOrderOfSubspaces(V)
subspaces = tuple(order.sorted())

distribution = Counter(
    (tables.subspace_dimensions[X], M.rank(X))
    for X in range(len(M.rank_table))
)

## Automorphism groups

- The order, group structure, and number of generators are displayed for both the projective and linear automorphism groups.
- The generators of each group are displayed in the following form:
  - Projective automorphism group: permutations of one-dimensional subspace IDs.
  - Linear automorphism group: matrices in $\mathrm{GL}(n,q)$ acting on row spaces by right multiplication.
- For $n>0$, the linear automorphism group contains the nonzero scalar matrices, and its order is $q-1$ times the order of the projective automorphism group.

In [4]:
projective_group = M.projective_automorphism_group(tables)
linear_group = M.linear_automorphism_group(tables)
display(table(
    [
        ("PGL action", projective_group.order(), projective_group.structure_description(), len(projective_group.gens())),
        ("GL preimage", linear_group.order(), linear_group.structure_description(), len(linear_group.gens())),
    ],
    header_row=["group", "order", "structure", "number of generators"],
))

print("Projective generators acting on point IDs:")
display(table(
    list(enumerate(projective_group.gens(), start=1)),
    header_row=["generator", "permutation"],
))

print("Linear generators acting on row spaces by right multiplication:")
display(table(
    [
        (i, str([list(row) for row in g.matrix().rows()]))
        for i, g in enumerate(linear_group.gens(), start=1)
    ],
    header_row=["generator", "matrix"],
))

  group         order   structure                                         number of generators
├─────────────┼───────┼─────────────────────────────────────────────────┼──────────────────────┤
  PGL action    64512   (C2 x C2 x C2 x C2 x C2 x C2) : (S3 x PSL(3,2))   11
  GL preimage   64512   (C2 x C2 x C2 x C2 x C2 x C2) : (PSL(3,2) x S3)   11

Projective generators acting on point IDs:


  generator   permutation
├───────────┼─────────────────────────────────────────────────────────────────────────────────────────────────┤
  1           (67,225)(134,185)(254,314)(281,300)(323,358)(339,350)(363,373)(368,371)
  2           (67,281)(134,254)(185,314)(225,300)(323,368)(339,363)(350,373)(358,371)
  3           (67,358)(134,350)(185,339)(225,323)(254,373)(281,371)(300,368)(314,363)
  4           (67,373)(134,371)(185,368)(225,363)(254,358)(281,350)(300,339)(314,323)
  5           (16,56)(32,61)(43,64)(51,66)(67,363,254,323)(134,368,281,339)(185,371,300,350)(225,373,314,358)
  6           (16,368,225)(32,363,185)(43,373,134)(51,371,67)(56,339,314)(61,323,300)(64,358,281)(66,350,254)
  7           (1,4)(10,15)(32,51)(61,66)(134,225)(281,314)(339,358)(368,373)
  8           (1,2)(10,13)(32,43)(61,64)(134,185)(281,300)(339,350)(368,371)
  9           (1,13)(2,10)(32,64)(43,61)(134,300)(185,281)(339,371)(350,368)
  10          (1,10,13,2)(4,5)(32,61,64,43)(51,56)(134,281,300,185)

Linear generators acting on row spaces by right multiplication:


  generator   matrix
├───────────┼───────────────────────────────────────────────────────────────────────────────────────┤
  1           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [1, 1, 0, 0, 1]]
  2           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [1, 0, 1, 0, 1]]
  3           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [1, 1, 0, 1, 1]]
  4           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [1, 1, 1, 1, 1]]
  5           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 1, 1, 0], [0, 0, 1, 1, 1]]
  6           [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [1, 0, 1, 1, 1], [1, 1, 0, 1, 0]]
  7           [[1, 1, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]]
  8           [[0, 1, 0, 0, 0], [1, 0, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]]
  9           [[0, 1, 1, 0, 0], [1, 0, 1, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1

## Basis encoding

- This is the sequence obtained by listing the $k$-dimensional subspaces in the fixed order used in the paper and assigning 1 to each basis and 0 otherwise.
- A canonical representative is one for which this sequence is lexicographically smallest among isomorphic q-matroids.
- The basis encoding of the selected canonical representative and its length are displayed.

In [5]:
encoding = M.basis_encoding(tables)
print("".join(map(str, encoding)))
print(f"Encoding length: {len(encoding)}")

00000000000000000000000000000000000000000011111111000000111111110000011111111000011111111000111111110011111111011111111111111110000000000000000000000000000
Encoding length: 155
